# 03 — Fine-Tuned Evaluation, Baseline Comparison & Side-by-Side Showcase

Evaluates the fine-tuned `v2` LoRA adapter (`results_v2/finetuned_v2/adapter`) on the exact frozen **30-case `v2` baseline contract** under both `rp_1.00` (greedy) and `gen_rp_1.08` (generated-only penalty `1.08`), compares all runs side-by-side against the `v2` base model, and prints a readable qualitative showcase.

### 1. Install Dependencies
Installs the pinned Hugging Face, PEFT, Accelerate, and evaluation packages required for adapter inference and scoring.

In [1]:
%pip install -q --progress-bar on "transformers==4.57.6" "datasets>=3,<5" "peft==0.18.1" "accelerate>=1,<2" pandas huggingface_hub

Installed / verified versions: torch==2.14.1+cu130, transformers==4.57.6, datasets==4.8.5, peft==0.18.1, accelerate==1.15.0


### 2. Verify Adapter, Dataset & Frozen Baseline Contract
Confirms `results_v2/finetuned_v2/adapter` and the 30-case `v2` validation split match the frozen baseline contract SHA-256.

In [2]:
import os, json, hashlib, random, re, time, shutil, importlib.metadata
from pathlib import Path
from getpass import getpass
from collections import Counter
import pandas as pd
import torch
from datasets import load_from_disk
from huggingface_hub import get_token
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    LogitsProcessor,
    LogitsProcessorList,
)

def digest(value):
    return hashlib.sha256(json.dumps(value, ensure_ascii=False, sort_keys=True).encode()).hexdigest()

def file_sha(path):
    sha = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b""):
            sha.update(block)
    return sha.hexdigest()

def read_token():
    token = os.environ.get("HF_TOKEN") or get_token()
    if not token:
        token = getpass("Hugging Face read token: ").strip()
    if not token:
        raise RuntimeError("A Hugging Face token with accepted Gemma access is required.")
    os.environ["HF_TOKEN"] = token
    return token

def runtime_path(path, sentinel):
    path = Path(path).expanduser()
    candidates = [path]
    if not path.is_absolute():
        candidates += [parent / path for parent in Path.cwd().parents]
        candidates.append(Path("/content") / path)
    for candidate in candidates:
        if (candidate / sentinel).is_file():
            return candidate.resolve()
    raise FileNotFoundError(f"Kernel directory: {Path.cwd()}; missing {sentinel} under {path}.")

class GeneratedOnlyRepetitionPenalty(LogitsProcessor):
    """Applies standard repetition penalty only to generated post-prompt tokens."""
    def __init__(self, penalty: float, prompt_length: int):
        if penalty <= 0:
            raise ValueError("penalty must be > 0")
        self.penalty = float(penalty)
        self.prompt_length = int(prompt_length)

    def __call__(self, input_ids: torch.LongTensor, scores: torch.FloatTensor) -> torch.FloatTensor:
        if self.penalty == 1.0 or input_ids.shape[1] <= self.prompt_length:
            return scores
        gen_ids = input_ids[:, self.prompt_length :]
        if gen_ids.shape[0] == 1:
            gen_ids = torch.unique(gen_ids[0]).unsqueeze(0)
        score = torch.gather(scores, 1, gen_ids)
        score = torch.where(score < 0, score * self.penalty, score / self.penalty)
        return scores.scatter(1, gen_ids, score)

from peft import PeftModel

PREPARED_ROOT = runtime_path("prepared_clinical_dataset", "latest_v2.json")
BASELINE_DIR = runtime_path("results_v2/baseline", "comparison_contract.json")
FINETUNE_DIR = runtime_path("results_v2/finetuned_v2", "adapter/adapter_config.json")
OUTPUT_DIR = (Path("/content/results_v2") if Path("/content").is_dir() else Path("results_v2")) / "comparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

contract = json.loads((BASELINE_DIR / "comparison_contract.json").read_text(encoding="utf-8"))
assert json.loads((FINETUNE_DIR / "comparison_contract.json").read_text(encoding="utf-8")) == contract
run_cfg = json.loads((FINETUNE_DIR / "run_config.json").read_text(encoding="utf-8"))
assert run_cfg["comparison_contract_sha256"] == digest(contract)

DATASET_VERSION = contract["dataset_version"]
prepared_dir = PREPARED_ROOT / DATASET_VERSION
manifest = json.loads((prepared_dir / "manifest.json").read_text(encoding="utf-8"))
prepared = load_from_disk(str(prepared_dir / "dataset"))
cases = prepared["validation"].to_list()
assert [c["case_id"] for c in cases] == contract["case_ids"]
assert digest([c["prompt_text"] for c in cases]) == contract["prompt_sha256"]
assert importlib.metadata.version("transformers") == contract["transformers"]
assert torch.__version__ == contract["torch"]

MODEL_ID = contract["model"]
MODEL_REVISION = contract["base_model_revision"]
MAX_NEW_TOKENS = contract["max_new_tokens"]
SEED = contract["seed"]
DECODING_CONDITIONS = contract["decoding_conditions"]
device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float32
print(f"Verified v2 eval setup ({DATASET_VERSION}): {len(cases)} cases, adapter={FINETUNE_DIR/'adapter'}", flush=True)

Verified v2 eval setup (eeff5efa1619f198): 30 cases, adapter=/content/results_v2/finetuned_v2/adapter


### 3. Evaluation Metrics & Self-Checks
Defines the canonical JSON, schema-key, and strict/soft value evaluators identical to `01_baseline_eval.ipynb` and runs unit checks.

In [3]:
TOP_LEVEL_KEYS = (
    "visit motivation", "admission", "patient information", "patient medical history",
    "surgeries", "symptoms", "medical examinations", "diagnosis tests", "treatments", "discharge",
)
MISSING_VALUES = ("", "none", "null", "unknown", "not provided")

def no_duplicate_keys(pairs):
    result = {}
    for key, value in pairs:
        if key in result:
            raise ValueError("Duplicate JSON key: " + key)
        result[key] = value
    return result

def reject_constant(value):
    raise ValueError("Nonstandard JSON constant: " + value)

def load_object(text, strict=True):
    obj = json.loads(
        text, strict=strict, object_pairs_hook=no_duplicate_keys, parse_constant=reject_constant
    )
    if not isinstance(obj, dict):
        raise ValueError("Expected a JSON object")
    return obj

def strip_fences(text):
    text = text.strip()
    if text.startswith("```") and text.endswith("```"):
        return re.sub(r"^```(?:json)?\s*", "", text, count=1, flags=re.IGNORECASE)[:-3].strip()
    return text

def facts(value, path=""):
    """Multiset of (field path, normalized value); array order is ignored."""
    result = Counter()
    if isinstance(value, dict):
        for key, child in value.items():
            result.update(facts(child, f"{path}.{key}" if path else key))
    elif isinstance(value, list):
        for child in value:
            result.update(facts(child, path))
    elif value is not None:
        normalized = " ".join(str(value).casefold().split())
        if normalized not in MISSING_VALUES:
            result[(path, normalized)] += 1
    return result

def facts_by_path(value, path=""):
    """Mapping field path -> list of normalized non-missing values."""
    result = {}
    for (p, v), cnt in facts(value, path).items():
        result.setdefault(p, []).extend([v] * cnt)
    return result

def key_paths(value, prefix=""):
    result = set()
    if isinstance(value, dict):
        for key, child in value.items():
            path = f"{prefix}.{key}" if prefix else key
            result.add(path)
            result.update(key_paths(child, path))
    elif isinstance(value, list):
        for child in value:
            result.update(key_paths(child, prefix))
    return result

def _token_f1(a_str, b_str):
    ta = Counter(re.findall(r"[a-z0-9]+", a_str.casefold()))
    tb = Counter(re.findall(r"[a-z0-9]+", b_str.casefold()))
    na, nb = sum(ta.values()), sum(tb.values())
    if not na or not nb:
        return 0.0
    ov = sum((ta & tb).values())
    return (2.0 * ov) / (na + nb)

def soft_fact_scores(expected_obj, actual_obj):
    if actual_obj is None:
        return 0.0, 0.0, 0.0
    exp_map = facts_by_path(expected_obj)
    act_map = facts_by_path(actual_obj)
    n_exp = sum(len(v) for v in exp_map.values())
    n_act = sum(len(v) for v in act_map.values())
    if not n_exp or not n_act:
        return 0.0, 0.0, 0.0
    total_sim = 0.0
    for path, exp_vals in exp_map.items():
        act_vals = act_map.get(path, [])
        if not act_vals:
            continue
        pairs = []
        for i, ev in enumerate(exp_vals):
            for j, av in enumerate(act_vals):
                s = 1.0 if ev == av else _token_f1(ev, av)
                if s > 0:
                    pairs.append((s, i, j))
        pairs.sort(reverse=True)
        used_e, used_a = set(), set()
        for s, i, j in pairs:
            if i not in used_e and j not in used_a:
                used_e.add(i)
                used_a.add(j)
                total_sim += s
    prec = total_sim / n_act
    rec = total_sim / n_exp
    f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
    return prec, rec, f1

def count_duplicate_array_items(text):
    try:
        obj = load_object(strip_fences(text))
    except ValueError:
        return None
    dups = 0
    def walk(node):
        nonlocal dups
        if isinstance(node, list):
            seen = set()
            for item in node:
                key = json.dumps(item, ensure_ascii=False, sort_keys=True)
                if key in seen:
                    dups += 1
                seen.add(key)
            for item in node:
                walk(item)
        elif isinstance(node, dict):
            for v in node.values():
                walk(v)
    walk(obj)
    return dups

def max_word_5gram_repeat(text):
    words = re.findall(r"\S+", text.lower())
    if len(words) < 5:
        return 0
    counts = Counter(tuple(words[i:i+5]) for i in range(len(words) - 4))
    return max(counts.values()) if counts else 0

def evaluate_output(text, reference):
    expected = load_object(reference)
    raw_valid = True
    try:
        load_object(text)
    except ValueError:
        raw_valid = False
    try:
        actual = load_object(strip_fences(text))
        error = None
    except ValueError as exc:
        actual, error = None, str(exc)
    expected_facts = facts(expected)
    actual_facts = facts(actual)
    matches = sum((expected_facts & actual_facts).values())
    n_actual, n_expected = sum(actual_facts.values()), sum(expected_facts.values())
    precision = matches / n_actual if n_actual else 0.0
    recall = matches / n_expected if n_expected else 0.0
    soft_p, soft_r, soft_f1 = soft_fact_scores(expected, actual)
    return {
        "empty": not text.strip(),
        "raw_json_object_valid": raw_valid,
        "json_object_valid_after_fence_removal": actual is not None,
        "parse_error": error,
        "exact_top_level_keys": actual is not None and set(actual.keys()) == set(TOP_LEVEL_KEYS),
        "exact_key_match": actual is not None and key_paths(actual) == key_paths(expected),
        "nonmissing_generated_values": n_actual,
        "reference_values": n_expected,
        "matched_values": matches,
        "placeholder_only": actual is not None and n_actual == 0,
        "value_precision": precision,
        "value_recall": recall,
        "value_f1": 2 * precision * recall / (precision + recall) if precision + recall else 0.0,
        "soft_value_precision": soft_p,
        "soft_value_recall": soft_r,
        "soft_value_f1": soft_f1,
        "duplicate_array_items": count_duplicate_array_items(text),
        "max_5gram_repeat": max_word_5gram_repeat(text),
    }

def score_run(cases, outputs, diagnostics):
    records = []
    for case in cases:
        key = case["case_id"]
        text = outputs[key]
        records.append({
            "case_id": key,
            **evaluate_output(text, case["reference"]),
            "reached_budget": bool(diagnostics[key].get("reached_budget")),
            "generated_tokens": int(diagnostics[key]["generated_tokens"]),
            "generated": text,
            "reference": case["reference"],
        })
    frame = pd.DataFrame(records)
    summary = {"selected_cases": len(cases), "evaluated_cases": len(records)}
    for name in (
        "empty", "raw_json_object_valid", "json_object_valid_after_fence_removal",
        "exact_top_level_keys", "exact_key_match", "placeholder_only", "reached_budget",
    ):
        summary[name + "_cases"] = int(frame[name].sum())
    for name in (
        "value_precision", "value_recall", "value_f1",
        "soft_value_precision", "soft_value_recall", "soft_value_f1",
    ):
        summary["mean_" + name] = float(frame[name].mean())
    for key, col in (
        ("matched_values", "matched_values"),
        ("reference_values", "reference_values"),
        ("generated_values", "nonmissing_generated_values"),
    ):
        summary[key] = int(frame[col].sum())
    summary["mean_generated_tokens"] = float(frame["generated_tokens"].mean())
    summary["max_5gram_repeat_ge_3_cases"] = int((frame["max_5gram_repeat"] >= 3).sum())
    return frame, summary

# Metric self-checks
ref = '{"age":"29","symptoms":[{"name":"chest pain"}]}'
assert evaluate_output(ref, ref)["value_f1"] == 1.0
assert evaluate_output("```json\n" + ref + "\n```", ref)["raw_json_object_valid"] is False
assert evaluate_output("```json\n" + ref + "\n```", ref)["json_object_valid_after_fence_removal"] is True
assert evaluate_output('{"age":"None"}', ref)["placeholder_only"] is True
assert 0.0 < evaluate_output('{"age":"29","symptoms":[{"name":"severe chest pain"}]}', ref)["soft_value_f1"] < 1.0
print("Metric self-checks passed.")

Metric self-checks passed.


### 4. Fine-Tuned Model Inference (`rp_1.00` & `gen_rp_1.08`)
Loads the base model merged with the trained `v2` LoRA adapter in FP32 and runs greedy and `gen_rp_1.08` inference across all 30 validation cases.

In [4]:
token = read_token()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=MODEL_REVISION, token=token)
pad_id = tokenizer.pad_token_id or tokenizer.eos_token_id
random.seed(SEED)
torch.manual_seed(SEED)
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, revision=MODEL_REVISION, token=token, dtype=dtype).to(device)
model = PeftModel.from_pretrained(base_model, str(FINETUNE_DIR / "adapter"), is_trainable=False).eval()
eos = model.generation_config.eos_token_id
eos_ids = eos if isinstance(eos, list) else [eos]

for cond in DECODING_CONDITIONS:
    cond_dir = OUTPUT_DIR / f"finetuned__{cond['tag']}"
    cond_dir.mkdir(parents=True, exist_ok=True)
    gen_path = cond_dir / "generations.json"
    diag_path = cond_dir / "generation_diagnostics.json"
    generations = json.loads(gen_path.read_text(encoding="utf-8")) if gen_path.exists() else {}
    diagnostics = json.loads(diag_path.read_text(encoding="utf-8")) if diag_path.exists() else {}
    started = time.monotonic()
    for case in cases:
        key = case["case_id"]
        if generations.get(key, "").strip() and key in diagnostics:
            continue
        inputs = tokenizer(case["prompt_text"], return_tensors="pt", add_special_tokens=False).to(device)
        n_prompt = inputs.input_ids.shape[1]
        assert n_prompt + MAX_NEW_TOKENS <= model.config.max_position_embeddings
        with torch.inference_mode():
            logits = model(**inputs, logits_to_keep=1).logits[:, -1, :]
            finite = bool(torch.isfinite(logits).all().item())
            del logits
            if not finite:
                ids = []
            else:
                gen_kwargs = {"max_new_tokens": MAX_NEW_TOKENS, "do_sample": False, "pad_token_id": pad_id}
                if cond["generated_only"] and cond["repetition_penalty"] != 1.0:
                    gen_kwargs["logits_processor"] = LogitsProcessorList([
                        GeneratedOnlyRepetitionPenalty(cond["repetition_penalty"], n_prompt)
                    ])
                generated = model.generate(**inputs, **gen_kwargs)
                ids = generated[0, n_prompt:].tolist()
        text = tokenizer.decode(ids, skip_special_tokens=True)
        diagnostics[key] = {
            "prompt_tokens": n_prompt,
            "generated_tokens": len(ids),
            "finite_first_token_logits": finite,
            "empty_output": not text.strip(),
            "reached_budget": len(ids) >= MAX_NEW_TOKENS,
            "ended_with_eos": bool(ids) and ids[-1] in eos_ids,
        }
        diag_path.write_text(json.dumps(diagnostics, ensure_ascii=False, indent=2), encoding="utf-8")
        if not finite or not text.strip():
            raise RuntimeError(f"finetuned/{cond['tag']}/{key}: failed generation")
        generations[key] = text
        gen_path.write_text(json.dumps(generations, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"[finetuned_v2 | {cond['tag']}] {len(generations)}/{len(cases)} {key}: {len(ids)} toks ({time.monotonic()-started:.1f}s)", flush=True)

del model, base_model
if device == "cuda":
    torch.cuda.empty_cache()

[finetuned_v2 | rp_1.00] 30/30 cases generated | mean=667.3 toks | min=330 | max=2048 | budget_hits=3/30
[finetuned_v2 | gen_rp_1.08] 30/30 cases generated | mean=604.9 toks | min=304 | max=2048 | budget_hits=2/30


### 5. Base vs Fine-Tuned 4-Way Comparison Table
Scores both base and fine-tuned runs across both decoding settings, saves `summary_comparison.csv` and `case_comparison.csv`, and displays the full 4-way table.

In [5]:
summary_rows = []
frames = {}

for cond in DECODING_CONDITIONS:
    tag = cond["tag"]
    b_dir = BASELINE_DIR / tag
    b_gens = json.loads((b_dir / "generations.json").read_text(encoding="utf-8"))
    b_diags = json.loads((b_dir / "generation_diagnostics.json").read_text(encoding="utf-8"))
    b_df, b_sum = score_run(cases, b_gens, b_diags)
    frames[f"base__{tag}"] = b_df
    summary_rows.append({"model": "base", "decoding": tag, **b_sum})

    f_dir = OUTPUT_DIR / f"finetuned__{tag}"
    f_gens = json.loads((f_dir / "generations.json").read_text(encoding="utf-8"))
    f_diags = json.loads((f_dir / "generation_diagnostics.json").read_text(encoding="utf-8"))
    f_df, f_sum = score_run(cases, f_gens, f_diags)
    f_df.to_csv(f_dir / "per_case.csv", index=False)
    (f_dir / "aggregate.json").write_text(json.dumps(f_sum, indent=2), encoding="utf-8")
    frames[f"finetuned_v2__{tag}"] = f_df
    summary_rows.append({"model": "finetuned_v2", "decoding": tag, **f_sum})

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(OUTPUT_DIR / "summary_comparison.csv", index=False)
(OUTPUT_DIR / "summary_comparison.json").write_text(json.dumps(summary_rows, indent=2), encoding="utf-8")

# Side-by-side per-case table for primary decoding (best valid finetuned decoding)
ft_runs = [r for r in summary_rows if r["model"] == "finetuned_v2"]
best_tag = max(ft_runs, key=lambda r: (r["raw_json_object_valid_cases"], r["mean_value_f1"]))["decoding"]
base_best = frames[f"base__{best_tag}"].rename(columns=lambda c: c if c in ("case_id", "reference") else f"base_{c}")
ft_best = frames[f"finetuned_v2__{best_tag}"].drop(columns=["reference"]).rename(columns=lambda c: c if c == "case_id" else f"ft_{c}")
case_comp = base_best.merge(ft_best, on="case_id", validate="one_to_one")
case_comp.to_csv(OUTPUT_DIR / "case_comparison.csv", index=False)
display(summary_df)
print("Best decoding tag for finetuned_v2:", best_tag, flush=True)

,model,decoding,selected_cases,evaluated_cases,empty_cases,raw_json_object_valid_cases,json_object_valid_after_fence_removal_cases,exact_top_level_keys_cases,exact_key_match_cases,placeholder_only_cases,reached_budget_cases,mean_value_precision,mean_value_recall,mean_value_f1,mean_soft_value_precision,mean_soft_value_recall,mean_soft_value_f1,matched_values,reference_values,generated_values,mean_generated_tokens,max_5gram_repeat_ge_3_cases
0,base,rp_1.00,30,30,0,0,6,0,0,0,10,0.005878,0.005529,0.005119,0.010737,0.008541,0.008710,4,1082,129,1240.333333,26
1,finetuned_v2,rp_1.00,30,30,0,25,25,25,0,0,3,0.179456,0.192002,0.180165,0.294161,0.316496,0.296107,193,1082,916,667.266667,21
2,base,gen_rp_1.08,30,30,0,0,22,0,0,0,2,0.007741,0.004103,0.004929,0.032294,0.017125,0.020850,4,1082,509,764.766667,21
3,finetuned_v2,gen_rp_1.08,30,30,0,28,28,28,0,0,2,0.188774,0.200838,0.189321,0.307967,0.323289,0.307384,198,1082,1001,604.900000,18


Best decoding tag for finetuned_v2: gen_rp_1.08


### 6. Qualitative Side-by-Side Showcase & Archive
Displays real side-by-side clinical note, reference JSON, base output, and fine-tuned output excerpts (`showcase_examples.md`) and writes `results_v2/comparison.zip`.

In [6]:
# Qualitative side-by-side showcase (saved to showcase_examples.md)
notes_by_id = {c["case_id"]: c["note"] for c in cases}
sorted_cases = case_comp.sort_values(by=["ft_value_f1", "ft_soft_value_f1"], ascending=False)
picks = []
for row in [sorted_cases.iloc[0], sorted_cases.iloc[1], sorted_cases.iloc[len(sorted_cases) // 2], sorted_cases.iloc[-1]]:
    if row["case_id"] not in [p["case_id"] for p in picks]:
        picks.append(row)

lines = [f"# Side-by-Side Showcase (`{best_tag}`)\n"]
for r in picks:
    cid = r["case_id"]
    lines.append(f"## {cid}")
    lines.append(f"- **Base:** raw_valid={r['base_raw_json_object_valid']}, exact_F1={r['base_value_f1']:.3f}, soft_F1={r['base_soft_value_f1']:.3f}, toks={r['base_generated_tokens']}")
    lines.append(f"- **Fine-tuned v2:** raw_valid={r['ft_raw_json_object_valid']}, exact_keys={r['ft_exact_key_match']}, exact_F1={r['ft_value_f1']:.3f}, soft_F1={r['ft_soft_value_f1']:.3f}, toks={r['ft_generated_tokens']}\n")
    lines.append(f"**Clinical Note (excerpt):**\n```\n{notes_by_id[cid][:600]}...\n```")
    lines.append(f"**Reference JSON:**\n```json\n{r['reference'][:1200]}\n```")
    lines.append(f"**Base Output:**\n```\n{str(r['base_generated'])[:900]}\n```")
    lines.append(f"**Fine-Tuned v2 Output:**\n```json\n{str(r['ft_generated'])[:1200]}\n```\n")

showcase_md = "\n".join(lines)
(OUTPUT_DIR / "showcase_examples.md").write_text(showcase_md, encoding="utf-8")
archive = shutil.make_archive(str(OUTPUT_DIR.resolve()), "zip", root_dir=OUTPUT_DIR.parent, base_dir=OUTPUT_DIR.name)
print(showcase_md[:3500])
print("\nSaved comparison archive:", archive, flush=True)

# Side-by-Side Showcase (`gen_rp_1.08`)

## validation:1161
- **Base:** raw_valid=False, exact_F1=0.000, soft_F1=0.038, toks=904
- **Fine-tuned v2:** raw_valid=True, exact_keys=False, exact_F1=0.508, soft_F1=0.566, toks=426

**Clinical Note (excerpt):**
```
A 49-year-old male patient sought treatment due to “repeated cough and sputum for one year and aggravation with chest tightness for one week.” Chest computed tomography (CT) in a local hospital revealed encapsulated effusion in the right thoracic cavity. Chest-enhanced CT after admission revealed a space-occupying lesion in the right chest. The pathological diagnosis according to percutaneous lung biopsy was a solitary fibrous tumor. Preoperative three-dimensional reconstruction showed that the blood supply to the tumor originated from the arteries and veins of the right lower lobe (Fig. ). Af...
```
**Reference JSON:**
```json
{"visit motivation":"repeated cough and sputum for one year and aggravation with chest tightness for one w